# FIB-Fusion on 4 real datasets: Federated IBML with Missing Modalities

Runs **paper-faithful IBML** (Qin et al., TPAMI 2026: BIO = Eq. 12–13, TCM = Eq. 15–16) in a **federated, missing-modality** setting on the four real datasets from your earlier notebook:

| Dataset | Task | Train / Val / Test | Text / Audio / Vision dims |
|---|---|---|---|
| CMU-MOSI | sentiment (neutral `0` dropped, `>0` positive) | 1284 / 229 / 686 | 300 / 5 / 20 |
| CMU-MOSEI | sentiment (neutral `0` dropped, `>0` positive) | 16265 / 1869 / 4643 | 300 / 74 / 35 |
| UR-FUNNY | humour (binary) | 8074 / 1034 / 1058 | 300 / 81 / 371 |
| MUStARD | sarcasm (binary) | 414 / 138 / 138 | 300 / 81 / 371 |

(Sizes are those reported in your earlier notebook; MOSI/MOSEI counts shrink slightly after dropping neutral samples. The loader prints the real numbers and warns if dimensions differ.)

### How to run
1. **Smoke test first:** set `QUICK = True` in the next cell and run all. It uses 1 seed and 5 rounds and finishes quickly, so you can catch any problem early.
2. **Full run:** set `QUICK = False`. The loop `DATASETS_TO_RUN` processes MUStARD → MOSI → UR-FUNNY → MOSEI. Each dataset writes to `/content/fib_runs/<dataset>/`, and finished datasets are skipped when you re-run (set `FORCE_RERUN = True` to redo).
3. **MOSEI** needs a high-RAM runtime (~3.5 GB pickle) and is by far the slowest. If Colab disconnects, set `USE_DRIVE = True` so results survive, and run MOSEI in its own session by setting `DATASETS_TO_RUN = ["MOSEI"]`.

### What is paper-faithful and what is not
| Element | Status |
|---|---|
| Decoupled logits `g = Σ φ_m W_m + b`, `g_m = φ_m W_m + b/M` (Eq. 1) | faithful (`model_type="linear"`) |
| `L_mib = Î·CE_f + Σ I_m (CE_f + CE_m)`, `ρ_m = Σ_k σ(g_m)_y`, `I_m = 1[ρ_m < mean ρ]` (Eq. 12–13) | faithful |
| TCM: Gaussian noise on **raw inputs**, strong modalities, correctly-classified samples, `p_m = min(Mρ̃_m/Σρ̃, λ)/λ`, `λ=10`, `ρ̃` from recorded previous-epoch predictions | faithful |
| SGD (momentum 0.9, wd 1e-4) + larger-LR warm-up | faithful in form; LR chosen on **validation** (see `tune_lr`) |
| Encoders | small MLPs on pre-extracted, time-pooled features (paper: ResNet18 / BiGRU / DGCNN) → **adapted** |
| Federated learning, non-IID clients, missing modalities, modality-aware aggregation | **not in the paper**: your extension |

**Pooling:** unlike the earlier notebook, time-mean pooling here **ignores all-zero (padded) time steps**. A real time step that is exactly zero in every feature is also ignored; that is an accepted simplification.
**Not executed by the author:** the data loader was tested with a stand-in dataset of the same layout; the training code was only syntax-checked. Run the unit checks (they run automatically per dataset) and the `QUICK` pass before trusting any number.

**v3 changes.** (1) FedProx strength is now read from the live config (the earlier notebook silently used μ=0, so `A1b_fedprox` equalled `A1_fedavg`); a unit check enforces it. (2) New variants `A1c_modagg_fedprox` and **`A7_fedprox_ibml` (FedProx + IBML)**. (3) The best method is chosen by **validation** score (`ranking_by_validation.csv`), never by test. (4) The learning rate is tuned in the federated setting (`tune_regime="federated"`), because a centralized choice can be too small for tiny clients. (5) Results from older runs are not reused (`DONE_v4` marker).

**v4 changes.** (1) The FedProx μ is **tuned on validation** over {0.001, 0.01, 0.1, 1.0} (μ=0 is evaluated for comparison, and the notebook says so if it wins). (2) MUStARD files with labels −1/+1 are mapped to 0/1 (other label values still raise an error). (3) Per-client sample counts and class mix are printed and saved (`client_balance_seed*.csv`), with warnings for tiny or single-class clients. (4) A second partition, `partition="balanced"` (equal client sizes, label skew only), is added to the sensitivity sweep, which now includes A7. (5) Reports now include weighted-F1 and per-class recall. (6) New checks: BIO equals CE for single-modality clients, partitions cover the training set exactly once, label mapping. (7) MOSI and MUStARD use 5 seeds.

## 0. Setup, switches and configuration

In [ ]:
import os, io, json, copy, time, random, pickle, gc, subprocess, sys
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             precision_recall_fscore_support)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODS = ["T", "A", "V"]
NC = 2                                   # binary tasks
print("device:", DEVICE)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# ---------------------------- switches ----------------------------
QUICK = False                            # True = 1 seed, 5 rounds, no LR tuning, no sensitivity (smoke test)
USE_DRIVE = False                        # True = keep data + results on Google Drive (survives disconnects)
FORCE_RERUN = False                      # True = ignore finished datasets and recompute
DATASETS_TO_RUN = ["MUSTARD", "MOSI", "UR_FUNNY", "MOSEI"]   # smallest first
# If Google Drive blocks a download ("Too many users have viewed or downloaded this file"):
# open the file's link in your browser -> right-click -> "Make a copy" into YOUR Drive, then point to the copy, e.g.
# MANUAL_PATHS = {"MOSI": "/content/drive/MyDrive/mosi_data.pkl"}
MANUAL_PATHS = {}

if USE_DRIVE or any(p.startswith("/content/drive") for p in MANUAL_PATHS.values()):
    from google.colab import drive
    drive.mount("/content/drive")
if USE_DRIVE:
    ROOT = "/content/drive/MyDrive/fib_fusion"
else:
    ROOT = "/content"
DATA_ROOT = f"{ROOT}/multimodal_data"
ROOT_OUT = f"{ROOT}/fib_runs"
os.makedirs(ROOT_OUT, exist_ok=True)

BASE_CFG = dict(
    # ---- clients ----
    n_clients=10,
    alpha=0.5,                        # Dirichlet concentration (0.1 strong non-IID, 100 ~ IID)
    min_client_size=16,
    client_patterns=["TAV", "TAV", "TA", "TV", "AV", "T", "T", "A", "V", "TA"],   # cycled over clients
    # ---- model ----
    model_type="linear",              # "linear" (paper-faithful, decoupled g_m) | "mlp" (adapted)
    hidden=128, d=64, dropout=0.2,
    # ---- optimisation (paper: SGD, momentum 0.9, wd 1e-4, warm-up with larger LR) ----
    optimizer="sgd", lr=0.01, momentum=0.9, weight_decay=1e-4,
    warmup_rounds=3, warmup_mult=3.0, grad_clip=5.0,      # grad_clip=None disables (not in the paper; stability guard)
    rounds=30, local_epochs=2, batch_size=32, client_frac=1.0,
    tune_lr=True, lr_grid=[0.003, 0.01, 0.03, 0.1, 0.3],  # LR chosen on VALIDATION with plain CE (no IBML, no FedProx)
    partition="dirichlet",            # "dirichlet" (label + size skew) | "balanced" (equal client sizes, label skew only)
    tune_mu=True, mu_grid=[0.0, 0.001, 0.01, 0.1, 1.0],   # FedProx mu chosen on VALIDATION (positive values; 0.0 shown for comparison)
    tune_regime="federated",          # "federated": tune inside the FL setup actually used | "central": one centralized client
    # ---- IBML ----
    bio_mode="paper",                 # "paper": L = L_mib (Eq.12) | "plus_ce": L = CE_f + lambda_bio * L_mib
    lambda_bio=0.5,                   # only used when bio_mode == "plus_ce"
    tcm_lambda=10.0,                  # paper: lambda = 10
    # ---- FL / reporting ----
    prox_mu=0.01, f1_threshold=0.60, seeds=[0, 1, 2],
    run_sensitivity=True,
    clip_std=10.0,                    # clip standardised features to +-10 (guards against outlier-driven blow-ups)
    key_map=dict(text=None, audio=None, vision=None, label=None),   # None = auto-detect
)

DATASETS = {
    "MOSI":     dict(file_id="1_XdzdW8UNG1TTS6QcX10uhoS6N11OBit", path=f"{DATA_ROOT}/MOSI/mosi_data.pkl",
                     label_mode="sentiment", expected_dims=dict(T=300, A=5, V=20),
                     folder="https://drive.google.com/drive/folders/1uEK737LXB9jAlf9kyqRs6B9N6cDncodq", cfg=dict(seeds=[0, 1, 2, 3, 4])),   # small: 5 seeds
    "MOSEI":    dict(file_id="180l4pN6XAv8-OAYQ6OrMheFUMwtqUWbz", path=f"{DATA_ROOT}/MOSEI/mosei_senti_data.pkl",
                     label_mode="sentiment", expected_dims=dict(T=300, A=74, V=35),
                     folder="https://drive.google.com/drive/folders/1A_hTmifi824gypelGobgl2M-5Rw9VWHv",
                     cfg=dict(batch_size=64, run_sensitivity=False)),           # large: bigger batches, skip sensitivity sweep
    "UR_FUNNY": dict(file_id="1L5slPmYyhEVtwGyM1kgcFMjeBpXLZGT0", path=f"{DATA_ROOT}/UR_FUNNY/humor.pkl",
                     label_mode="binary", expected_dims=dict(T=300, A=81, V=371),
                     folder="https://drive.google.com/drive/folders/1Agzm157lciMONHOHemHRSySmjn1ahHX1", cfg={}),
    "MUSTARD":  dict(file_id="1EMBUmUL5B0PTncGx3L-sBElGOmjFBR_h", path=f"{DATA_ROOT}/MUStARD/sarcasm.pkl",
                     label_mode="binary", expected_dims=dict(T=300, A=81, V=371),
                     folder="https://drive.google.com/drive/folders/1JFcX-NF97zu9ZOZGALGU9kp8dwkP7aJ7",
                     cfg=dict(batch_size=16, seeds=[0, 1, 2, 3, 4])),                                  # tiny: ~40 samples per client
}
for _n, _p in MANUAL_PATHS.items():
    DATASETS[_n]["path"] = _p
for _n in DATASETS_TO_RUN:
    assert _n in DATASETS, f"unknown dataset {_n}; choose from {list(DATASETS)}"

CFG = {}                                 # filled (in place) per dataset by use_dataset()
DATA = DIMS = OUT_DIR = None             # set per dataset by use_dataset()

## 1. Download and prepare a dataset

Downloads only the processed pickle (not the multi-GB raw files). If a direct download hits Google Drive's quota, use the folder URL stored in `DATASETS[name]["folder"]`.
Each pickle holds `{"train", "valid", "test"}` splits with `text`, `audio`, `vision`, `labels`. Sequences `(N, L, D)` are mean-pooled over **non-padded** time steps (in chunks, to keep RAM low). Normalisation is fitted on **train only** and clipped to ±`clip_std`. The loader refuses arbitrary classes while unpickling.

In [ ]:
try:
    import gdown
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown

def download_dataset(name):
    spec = DATASETS[name]
    os.makedirs(os.path.dirname(spec["path"]), exist_ok=True)
    if os.path.exists(spec["path"]) and os.path.getsize(spec["path"]) > 0:
        print(f"{name}: already present ({os.path.getsize(spec['path']) / 1024**3:.2f} GB)")
        return spec["path"]
    try:
        out = gdown.download(id=spec["file_id"], output=spec["path"], quiet=False)
    except Exception as e:
        out = None; print(f"gdown error: {str(e)[:200]}")
    if not out or not os.path.exists(spec["path"]):
        raise RuntimeError(f"Download failed for {name} (Google Drive quota is the usual cause). Fix: open "
                           f"https://drive.google.com/file/d/{spec['file_id']}/view in your browser, choose 'Make a copy' "
                           f"into your own Drive, then set MANUAL_PATHS = {{'{name}': '/content/drive/MyDrive/<copied file>'}} "
                           f"in the setup cell and re-run. (Folder alternative: {spec['folder']})")
    print(f"{name}: downloaded {os.path.getsize(spec['path']) / 1024**3:.2f} GB -> {spec['path']}")
    return spec["path"]

ALLOWED = ("numpy", "builtins", "collections", "_codecs", "pandas")
class SafeUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module.split(".")[0] in ALLOWED:
            return super().find_class(module, name)
        raise pickle.UnpicklingError(f"Blocked global: {module}.{name}")

ALIASES = dict(
    text=["text", "text_bert", "language", "glove", "words"],
    audio=["audio", "acoustic", "covarep"],
    vision=["vision", "visual", "video", "facet"],
    label=["regression_labels", "labels", "label", "sentiment", "annotations"],
)
SPLIT_ALIASES = dict(train=["train"], val=["valid", "val", "dev"], test=["test"])

def _pick(d, aliases, override):
    if override is not None:
        return d[override]
    for k in aliases:
        if k in d:
            return d[k]
    raise KeyError(f"None of {aliases} found. Available keys: {list(d.keys())}. Set CFG['key_map'] explicitly.")

def _pool(a, name="", chunk=2048):
    # (N, L, D) -> (N, D): mean over non-padded steps (a step with all-zero features counts as padding).
    # Done in chunks so peak RAM stays small even for MOSEI.
    a = np.asarray(a)
    if a.ndim == 2:
        return np.nan_to_num(a.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    if a.ndim != 3:
        raise ValueError(f"Unsupported feature shape {a.shape} for modality {name}")
    if name == "T" and a.shape[1] == 3:
        print("WARNING: text array has shape (N,3,L); this looks like BERT [ids, mask, segment], not features.")
    out = np.empty((a.shape[0], a.shape[2]), dtype=np.float32)
    for s in range(0, a.shape[0], chunk):
        x = np.nan_to_num(a[s:s + chunk].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        cnt = np.maximum((np.abs(x).sum(-1) > 0).sum(1, keepdims=True), 1)
        out[s:s + chunk] = x.sum(1) / cnt                      # padded steps are exactly zero, so they add nothing
    return out

def _to_labels(s, mode):
    s = np.asarray(s, dtype=np.float32).reshape(-1)
    if mode == "sentiment":                                     # MOSI / MOSEI: drop neutral, >0 positive
        keep = s != 0
        return keep, (s[keep] > 0).astype(np.int64)
    if not set(np.unique(s)).issubset({0.0, 1.0}):              # UR-FUNNY / MUStARD: must already be binary
        if set(np.unique(s).tolist()).issubset({-1.0, 1.0}):        # some MUStARD files use -1/+1: map -1 -> 0, +1 -> 1
            return np.ones_like(s, dtype=bool), (s > 0).astype(np.int64)
        raise ValueError(f"Expected binary labels in {{0,1}} or {{-1,1}}, got values {np.unique(s)[:10]}")
    return np.ones_like(s, dtype=bool), s.astype(np.int64)

def load_pkl(name, cfg):
    spec = DATASETS[name]
    with open(spec["path"], "rb") as f:
        raw = SafeUnpickler(f, encoding="latin1").load()
    print("Top-level keys:", list(raw.keys())[:20] if isinstance(raw, dict) else type(raw))
    km = cfg["key_map"]; out, excluded = {}, {}
    for split, names in SPLIT_ALIASES.items():
        key = next((n for n in names if n in raw), None)
        if key is None:
            raise KeyError(f"Split '{split}' not found in {list(raw.keys())}.")
        sd = raw.pop(key)                                       # pop + pool one split at a time (low peak RAM)
        X = {"T": _pool(_pick(sd, ALIASES["text"], km["text"]), "T"),
             "A": _pool(_pick(sd, ALIASES["audio"], km["audio"]), "A"),
             "V": _pool(_pick(sd, ALIASES["vision"], km["vision"]), "V")}
        keep, y = _to_labels(_pick(sd, ALIASES["label"], km["label"]), spec["label_mode"])
        for m in MODS:
            assert len(X[m]) == len(keep), f"{name}/{split}: {m} has {len(X[m])} rows but labels have {len(keep)}"
        out[split] = {m: X[m][keep] for m in MODS}
        out[split]["y"] = y
        excluded[split] = int((~keep).sum())
        del sd, X; gc.collect()
    del raw; gc.collect()
    print(f"[{name}] label_mode={spec['label_mode']} | excluded neutral samples per split: {excluded}")
    return out

def standardise(data, clip):
    # Fit normalisation on TRAIN only; constant train features get scale 1; clip outliers.
    for m in MODS:
        mu = data["train"][m].mean(0, keepdims=True)
        sd = data["train"][m].std(0, keepdims=True)
        sd = np.where(sd < 1e-6, 1.0, sd)
        for s in data:
            data[s][m] = np.clip((data[s][m] - mu) / sd, -clip, clip).astype(np.float32)
    return data

def to_device(data):
    return {s: {k: torch.as_tensor(v).to(DEVICE) for k, v in d.items()} for s, d in data.items()}

def load_dataset(name, cfg):
    data = standardise(load_pkl(name, cfg), cfg["clip_std"])
    exp = DATASETS[name]["expected_dims"]
    for s in data:
        y = data[s]["y"]
        dims = {m: data[s][m].shape[1] for m in MODS}
        print(f"{s:5s} n={len(y):6d}  pos={int(y.sum()):6d}  neg={int((1-y).sum()):6d}  dims={dims}")
        if s == "train" and dims != exp:
            print(f"  WARNING: dims differ from the expected {exp}; check key_map / file version.")
    return to_device(data)

def use_dataset(name):
    # Select a dataset: merge its overrides into CFG (in place), download, load, and set the globals.
    global DATA, DIMS, OUT_DIR
    CFG.clear(); CFG.update({**BASE_CFG, **DATASETS[name]["cfg"], "dataset": name})
    if QUICK:
        CFG.update(seeds=[0], rounds=5, tune_lr=False, tune_mu=False, run_sensitivity=False)
    download_dataset(name)
    DATA = None; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    DATA = load_dataset(name, CFG)
    DIMS = {m: DATA["train"][m].shape[1] for m in MODS}
    OUT_DIR = f"{ROOT_OUT}/{name}"; os.makedirs(OUT_DIR, exist_ok=True)
    print(f"dataset: {name} | outputs -> {OUT_DIR}")

## 2. Federated clients: non-IID labels and modality masks

In [ ]:
def dirichlet_partition(y, n_clients, alpha, min_size, rng, max_tries=500):
    classes = np.unique(y)
    idx = None
    for _ in range(max_tries):
        idx = [[] for _ in range(n_clients)]
        for c in classes:
            ids = np.where(y == c)[0]; rng.shuffle(ids)
            p = rng.dirichlet(np.repeat(alpha, n_clients))
            cuts = (np.cumsum(p) * len(ids)).astype(int)[:-1]
            for k, part in enumerate(np.split(ids, cuts)):
                idx[k] += part.tolist()
        if min(len(i) for i in idx) >= min_size:
            return idx
    for k in range(n_clients):                       # fallback: top up undersized clients
        while len(idx[k]) < min_size:
            big = max(range(n_clients), key=lambda j: len(idx[j]))
            idx[k].append(idx[big].pop())
    return idx

def balanced_size_partition(y, n_clients, alpha, rng):
    # Equal-size clients (sizes differ by at most 1) with Dirichlet label skew: every client gets ~N/K samples,
    # but its class mix is drawn from Dirichlet(alpha). Isolates label skew from client-size imbalance.
    N = len(y); classes = np.unique(y)
    pools = {c: list(rng.permutation(np.where(y == c)[0])) for c in classes}
    prior = np.array([(y == c).mean() for c in classes])
    sizes = [N // n_clients + (1 if k < N % n_clients else 0) for k in range(n_clients)]
    idx = [[] for _ in range(n_clients)]
    for k in rng.permutation(n_clients):
        q = rng.dirichlet(alpha * len(classes) * prior)         # centred on the dataset's class ratio (alpha -> inf = IID)
        want = np.floor(q * sizes[k]).astype(int)
        want[np.argmax(q)] += sizes[k] - want.sum()
        for ci, c in enumerate(classes):
            for _ in range(min(int(want[ci]), len(pools[c]))):
                idx[k].append(pools[c].pop())
        while len(idx[k]) < sizes[k]:                         # a class pool ran dry: fill from the largest remaining pool
            c = max(classes, key=lambda cc: len(pools[cc]))
            idx[k].append(pools[c].pop())
    return idx

def client_balance_report(clients, min_n=30, skew=0.9):
    # Per-client sample counts and class mix, with warnings for tiny or (almost) single-class clients.
    df = describe_clients(clients).copy()
    df["pos"] = (df["pos_frac"] * df["n"]).round().astype(int)
    df["neg"] = df["n"] - df["pos"]
    df["flag"] = ""
    df.loc[df["n"] < min_n, "flag"] += "small "
    df.loc[(df["pos_frac"] >= skew) | (df["pos_frac"] <= 1 - skew), "flag"] += "label-skewed"
    return df

def build_clients(data, cfg, seed):
    rng = np.random.default_rng(seed)
    y = data["train"]["y"].cpu().numpy()
    if cfg.get("partition", "dirichlet") == "balanced":
        parts = balanced_size_partition(y, cfg["n_clients"], cfg["alpha"], rng)
    else:
        parts = dirichlet_partition(y, cfg["n_clients"], cfg["alpha"], cfg["min_client_size"], rng)
    clients = []
    for k, ids in enumerate(parts):
        pat = cfg["client_patterns"][k % len(cfg["client_patterns"])]
        ids = sorted(ids)
        clients.append(dict(id=k, pattern=pat, mods=[m for m in MODS if m in pat],
                            mask=tuple(m in pat for m in MODS),
                            idx=torch.as_tensor(ids, device=DEVICE), n=len(ids),
                            pos_frac=float(y[ids].mean())))
    return clients

def describe_clients(clients):
    return pd.DataFrame([{k: c[k] for k in ("id", "pattern", "n", "pos_frac")} for c in clients])

## 3. Models

Every model returns `(fused_logits, {modality: g_m})` and only needs the **observed** modalities in `x`; absent modalities are never passed through their encoder.

* **`LinearIBMLNet` (paper-faithful, default).** `g_m = φ_m(x_m) W_m + b/|avail|` and `fused = Σ_m g_m`, exactly Eq. (1) with `M = |avail|`. An absent modality simply contributes no term, so the fused logits and the `g_m` stay decoupled for any mask.
* **`MLPFusionNet` (adapted).** Absent slots are zero-filled and a mask vector is appended; a non-linear fusion MLP produces the fused logits and `g_m` come from auxiliary linear heads. Useful as a stronger fusion model, but `g_m` are then *not* parts of the fused logits.

In [ ]:
def mlp_encoder(i, h, p):
    return nn.Sequential(nn.Linear(i, h), nn.ReLU(), nn.Dropout(p), nn.Linear(h, h), nn.ReLU())

class LinearIBMLNet(nn.Module):
    def __init__(self, dims, hidden=128, p=0.2):
        super().__init__()
        self.enc = nn.ModuleDict({m: mlp_encoder(dims[m], hidden, p) for m in MODS})
        self.W = nn.ModuleDict({m: nn.Linear(hidden, NC, bias=False) for m in MODS})
        self.bias = nn.Parameter(torch.zeros(NC))            # shared parameter

    def forward(self, x, mask):
        avail = [m for i, m in enumerate(MODS) if mask[i]]
        mod = {m: self.W[m](self.enc[m](x[m])) + self.bias / len(avail) for m in avail}
        return sum(mod.values()), mod

class MLPFusionNet(nn.Module):
    def __init__(self, dims, hidden=128, d=64, p=0.2):
        super().__init__()
        self.d = d
        self.enc = nn.ModuleDict({m: mlp_encoder(dims[m], hidden, p) for m in MODS})
        self.proj = nn.ModuleDict({m: nn.Linear(hidden, d) for m in MODS})
        self.head = nn.ModuleDict({m: nn.Linear(d, NC) for m in MODS})
        self.fusion = nn.Sequential(nn.Linear(3 * d + 3, hidden), nn.ReLU(), nn.Dropout(p),
                                    nn.Linear(hidden, d), nn.ReLU())
        self.cls = nn.Linear(d, NC)

    def forward(self, x, mask):
        first = next(m for i, m in enumerate(MODS) if mask[i])
        B, dev = x[first].shape[0], x[first].device
        zs, mod = [], {}
        for i, m in enumerate(MODS):
            if mask[i]:
                z = self.proj[m](self.enc[m](x[m])); mod[m] = self.head[m](z); zs.append(z)
            else:
                zs.append(torch.zeros(B, self.d, device=dev))
        mvec = torch.tensor([float(v) for v in mask], device=dev).expand(B, 3)
        return self.cls(self.fusion(torch.cat(zs + [mvec], dim=1))), mod

def build_model(cfg):
    if cfg["model_type"] == "linear":
        return LinearIBMLNet(DIMS, cfg["hidden"], cfg["dropout"]).to(DEVICE)
    return MLPFusionNet(DIMS, cfg["hidden"], cfg["d"], cfg["dropout"]).to(DEVICE)

## 4. IBML local objective: BIO (Eq. 12–13) and TCM (Eq. 15–16)

**BIO.** For a batch, `ρ_m = Σ_k σ(g_m(x_k))_{y_k}` (detached), `I_m = 1[ρ_m < mean_m ρ_m]`, `Î = max(1 − Σ I_m, 0)` and

`L_mib = Î·CE(fused) + Σ_m I_m·( CE(fused) + CE(g_m) )`.

With a single available modality `L_mib` reduces to plain CE (no cross-modal term is invented).

**TCM.** Each client keeps a *record* of the last prediction made for each of its training samples (`p_m(y)` and whether `g_m` was correct). Before a batch is processed, `ρ̃_m` is summed from these records; every modality with `ρ̃_m` above average gets Gaussian noise `N(0, var(x_m))` added to its **raw inputs**, only for samples that `g_m` classified correctly, each with probability `p_m = min(M ρ̃_m / Σ ρ̃, λ)/λ`. Records are client-side state, so no extra communication is needed.

In [ ]:
def local_loss(model, xb, yb, mask, cfg, use_bio):
    avail = [m for i, m in enumerate(MODS) if mask[i]]
    lf, lm = model(xb, mask)
    ce_f = F.cross_entropy(lf, yb)
    B = len(yb); idx = torch.arange(B, device=yb.device)
    P = torch.zeros(B, 3, device=yb.device)                    # p_m(y) per sample, for TCM records
    OK = torch.zeros(B, 3, dtype=torch.bool, device=yb.device) # g_m correct per sample
    with torch.no_grad():
        for m in avail:
            i = MODS.index(m)
            P[:, i] = F.softmax(lm[m], 1)[idx, yb]
            OK[:, i] = lm[m].argmax(1) == yb
    info = dict(P=P, OK=OK, I=None)
    if not use_bio or len(avail) < 2:
        return ce_f, info
    rho = {m: P[:, MODS.index(m)].sum() for m in avail}         # Eq. (13), no gradient
    rbar = sum(rho.values()) / len(avail)
    I = {m: float(rho[m] < rbar) for m in avail}
    I_hat = max(1.0 - sum(I.values()), 0.0)
    bio = I_hat * ce_f
    for m in avail:
        if I[m]:
            bio = bio + ce_f + F.cross_entropy(lm[m], yb)
    info["I"] = I
    loss = bio if cfg["bio_mode"] == "paper" else ce_f + cfg["lambda_bio"] * bio
    return loss, info

def new_record(n_train):
    return dict(p=torch.zeros(n_train, 3, device=DEVICE),
                ok=torch.zeros(n_train, 3, dtype=torch.bool, device=DEVICE),
                valid=torch.zeros(n_train, dtype=torch.bool, device=DEVICE))

def apply_tcm(xb, b, avail, rec, cfg):
    # xb: dict of raw-input batches; b: global train indices of the batch; rec: this client's record.
    valid = rec["valid"][b]
    if not bool(valid.any()) or len(avail) < 2:
        return xb
    P = rec["p"][b] * valid[:, None].float()
    rho = {m: P[:, MODS.index(m)].sum() for m in avail}
    tot = sum(rho.values())
    if float(tot) <= 0:
        return xb
    M, lam = len(avail), cfg["tcm_lambda"]
    mean = tot / M
    out = dict(xb)
    for m in avail:
        if rho[m] > mean:                                       # strong modality
            i = MODS.index(m)
            p_m = min(float(M * rho[m] / tot), lam) / lam
            sel = valid & rec["ok"][b][:, i] & (torch.rand(len(b), device=DEVICE) < p_m)
            if bool(sel.any()):
                std = xb[m].std(0, keepdim=True)
                out[m] = xb[m] + sel[:, None].float() * torch.randn_like(xb[m]) * std
    return out

## 5. Federated training: local update, modality-aware aggregation, evaluation

In [ ]:
PARAM_MOD_PREFIX = ("enc", "proj", "head", "W")

def key_modality(key):
    # Which modality a parameter belongs to (None = shared).
    parts = key.split(".")
    return parts[1] if parts[0] in PARAM_MOD_PREFIX else None

def clone_sd(sd): return {k: v.detach().clone() for k, v in sd.items()}

def make_opt(model, cfg, lr):
    if cfg["optimizer"] == "sgd":
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=cfg["momentum"], weight_decay=cfg["weight_decay"])
    return torch.optim.Adam(model.parameters(), lr=lr)

def round_lr(cfg, r, warmup=True):
    # Larger LR in the first rounds (paper's warm-up), then the base LR.
    return cfg["lr"] * (cfg["warmup_mult"] if (warmup and r <= cfg["warmup_rounds"]) else 1.0)

def local_train(global_sd, client, data, cfg, use_bio, use_tcm, mu, lr, rec=None,
                epochs=None, model=None, opt=None):
    if model is None:
        model = build_model(cfg); model.load_state_dict(global_sd)
    model.train()
    if opt is None:
        opt = make_opt(model, cfg, lr)
    for g in opt.param_groups: g["lr"] = lr
    ref = {k: v.detach().clone() for k, v in global_sd.items()} if mu > 0 else None
    tr, idx = data["train"], client["idx"]
    avail = client["mods"]
    for _ in range(epochs or cfg["local_epochs"]):
        perm = idx[torch.randperm(len(idx), device=DEVICE)]
        for s in range(0, len(perm), cfg["batch_size"]):
            b = perm[s:s + cfg["batch_size"]]
            if len(b) < 2: continue
            xb = {m: tr[m][b] for m in avail}
            if use_tcm and rec is not None:
                xb = apply_tcm(xb, b, avail, rec, cfg)
            loss, info = local_loss(model, xb, tr["y"][b], client["mask"], cfg, use_bio)
            if rec is not None:                                    # record predictions for the next visit
                rec["p"][b] = info["P"]; rec["ok"][b] = info["OK"]; rec["valid"][b] = True
            if ref is not None:                                    # FedProx, applied once
                loss = loss + 0.5 * mu * sum(((p - ref[k]) ** 2).sum() for k, p in model.named_parameters()
                                             if key_modality(k) is None or key_modality(k) in avail)
            opt.zero_grad(); loss.backward()
            if cfg.get("grad_clip"): nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            opt.step()
    return model, opt

def aggregate(global_sd, results, mode, d):
    # results: list of dict(sd, n, mods). mode: 'fedavg' (all params, all clients) | 'modality' (modality-aware).
    new = {}
    for k, g in global_sd.items():
        mod = key_modality(k)
        def wavg(rs, sl=None):
            tot = sum(r["n"] for r in rs)
            return sum((r["sd"][k] if sl is None else r["sd"][k][:, sl]) * (r["n"] / tot) for r in rs)
        if mode == "fedavg":
            new[k] = wavg(results); continue
        if mod is not None:                                        # encoder / projection / head of one modality
            rs = [r for r in results if mod in r["mods"]]
            new[k] = wavg(rs) if rs else g
        elif k == "fusion.0.weight":                               # MLP model: column blocks tied to modalities
            w = g.clone()
            for i, m in enumerate(MODS):
                rs = [r for r in results if m in r["mods"]]
                sl = slice(i * d, (i + 1) * d)
                if rs: w[:, sl] = wavg(rs, sl)
            sl = slice(3 * d, 3 * d + 3)                           # mask-indicator columns: all clients
            w[:, sl] = wavg(results, sl)
            new[k] = w
        else:
            new[k] = wavg(results)                                 # shared parameters
    return new

def upload_numel(sd, mods, mode):
    if mode == "fedavg": return sum(v.numel() for v in sd.values())
    return sum(v.numel() for k, v in sd.items() if key_modality(k) is None or key_modality(k) in mods)

ALL_CONDS = ["TAV", "TA", "TV", "AV", "T", "A", "V"]

def metrics(y, p):
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, labels=[0, 1], zero_division=0)
    return dict(acc=accuracy_score(y, p), macro_f1=f1_score(y, p, average="macro", zero_division=0),
                weighted_f1=f1_score(y, p, average="weighted", zero_division=0),
                bal_acc=balanced_accuracy_score(y, p),
                prec_neg=pr[0], prec_pos=pr[1], rec_neg=rc[0], rec_pos=rc[1])

@torch.no_grad()
def predict(model, split, cond):
    model.eval()
    mask = tuple(m in cond for m in MODS)
    preds = []
    for s in range(0, len(split["y"]), 1024):
        xb = {m: split[m][s:s + 1024] for i, m in enumerate(MODS) if mask[i]}
        preds.append(model(xb, mask)[0].argmax(1))
    return torch.cat(preds).cpu().numpy()

def eval_conditions(model, split, conds=ALL_CONDS):
    y = split["y"].cpu().numpy()
    return {c: metrics(y, predict(model, split, c)) for c in conds}

def val_score(model, data):
    return float(np.mean([r["macro_f1"] for r in eval_conditions(model, data["val"]).values()]))

@torch.no_grad()
def modality_balance(model, split, cond="TAV"):
    # Discrepancy ratio max(rho)/min(rho) of the per-modality correct-class probabilities (1.0 = balanced).
    model.eval()
    mask = tuple(m in cond for m in MODS); y = split["y"]
    _, mod = model({m: split[m] for i, m in enumerate(MODS) if mask[i]}, mask)
    idx = torch.arange(len(y), device=y.device)
    rho = {m: float(F.softmax(mod[m], 1)[idx, y].mean()) for m in mod}
    acc = {m: float((mod[m].argmax(1) == y).float().mean()) for m in mod}
    return dict(rho=rho, acc=acc, ratio=max(rho.values()) / max(min(rho.values()), 1e-8))

In [ ]:
def run_federated(data, clients, cfg, agg="modality", use_bio=False, use_tcm=False, warmup=True,
                  mu=0.0, seed=0, name="", cfg_over=None):
    cfg = {**cfg, **(cfg_over or {})}
    if mu is None: mu = cfg["prox_mu"]      # late binding: FedProx strength is read from the live config
    set_seed(seed)
    model = build_model(cfg); gsd = clone_sd(model.state_dict())
    n_train = len(data["train"]["y"])
    recs = {c["id"]: new_record(n_train) for c in clients} if use_tcm else {}
    best = (-1, None, 0); hist = []; comm = 0; thr_round = None
    rng = np.random.default_rng(seed)
    for r in range(1, cfg["rounds"] + 1):
        k = max(1, int(round(cfg["client_frac"] * len(clients))))
        chosen = [clients[i] for i in rng.choice(len(clients), k, replace=False)]
        results = []
        for c in chosen:
            m_, _ = local_train(gsd, c, data, cfg, use_bio, use_tcm, mu, round_lr(cfg, r, warmup),
                                rec=recs.get(c["id"]))
            sd = clone_sd(m_.state_dict())
            results.append(dict(sd=sd, n=c["n"], mods=c["mods"]))
            comm += 4 * (sum(v.numel() for v in gsd.values()) + upload_numel(sd, c["mods"], agg))
        gsd = aggregate(gsd, results, agg, cfg["d"])
        model.load_state_dict(gsd)
        vs = val_score(model, data)
        hist.append(dict(round=r, val_score=vs))
        if thr_round is None and vs >= cfg["f1_threshold"]: thr_round = r
        if vs > best[0]: best = (vs, clone_sd(gsd), r)
    model.load_state_dict(best[1])
    return dict(name=name, seed=seed, best_round=best[2], best_val=best[0],
                test=eval_conditions(model, data["test"]), bal=modality_balance(model, data["test"]),
                history=hist, rounds_to_thr=thr_round, comm_MB=comm / 1e6)

def run_local_only(data, clients, cfg, use_bio=False, seed=0, name="A0_local"):
    # Each client trains alone; evaluated on test under ITS OWN modality pattern.
    set_seed(seed)
    per_client = []
    for c in clients:
        model = build_model(cfg); opt = None; best = (-1, None)
        for r in range(1, cfg["rounds"] + 1):
            model, opt = local_train(None, c, data, cfg, use_bio, False, 0.0, round_lr(cfg, r), model=model, opt=opt)
            vm = metrics(data["val"]["y"].cpu().numpy(), predict(model, data["val"], c["pattern"]))["macro_f1"]
            if vm > best[0]: best = (vm, clone_sd(model.state_dict()))
        model.load_state_dict(best[1])
        per_client.append(metrics(data["test"]["y"].cpu().numpy(), predict(model, data["test"], c["pattern"])))
    avg = {k: float(np.mean([p[k] for p in per_client])) for k in per_client[0]}
    return dict(name=name, seed=seed, best_round=cfg["rounds"], best_val=np.nan, test={"own_mask": avg},
                history=[], rounds_to_thr=None, comm_MB=0.0)

## 6. Unit checks (run once before trusting any result)

In [ ]:
def _run_unit_checks():
    cl_all = build_clients(DATA, CFG, 0)

    # 0) client partition is disjoint and covers the training set (no leakage between clients)
    all_idx = torch.cat([c["idx"] for c in cl_all]).cpu().numpy()
    assert len(all_idx) == len(np.unique(all_idx)) == len(DATA["train"]["y"]), "partition overlaps or misses samples"
    print("0) partition check passed")

    # 1) a client without audio must leave audio parameters untouched during local training (all variants)
    cl = [c for c in cl_all if "A" not in c["pattern"]][0]
    sd0 = clone_sd(build_model(CFG).state_dict())
    for use_bio, use_tcm, mu in [(False, False, 0.0), (True, True, 0.0), (True, False, 0.01), (True, True, 0.01)]:
        rec = new_record(len(DATA["train"]["y"])) if use_tcm else None
        m1, _ = local_train(sd0, cl, DATA, CFG, use_bio, use_tcm, mu, CFG["lr"], rec=rec)
        sd1 = m1.state_dict()
        assert not any((sd1[k] != sd0[k]).any().item() for k in sd0 if key_modality(k) == "A"), "audio params changed"
        assert any((sd1[k] != sd0[k]).any().item() for k in sd0 if key_modality(k) in ("T", "V", None))
    print("1) absent-modality parameters untouched (CE / BIO+TCM / FedProx) passed")

    # 2) weighted modality-aware aggregation equals the manual weighted average
    sd_a = {k: v + 1.0 if v.is_floating_point() else v for k, v in sd0.items()}
    sd_b = {k: v + 3.0 if v.is_floating_point() else v for k, v in sd0.items()}
    res = [dict(sd=sd_a, n=10, mods=MODS), dict(sd=sd_b, n=30, mods=MODS)]
    agg = aggregate(sd0, res, "modality", CFG["d"])
    assert all(torch.allclose(agg[k], sd0[k] + 2.5, atol=1e-5) for k in sd0), "weighted average wrong"
    # ... and keeps the global value for a modality nobody has
    res = [dict(sd=clone_sd(sd_a), n=10, mods=["T"])]
    agg = aggregate(sd0, res, "modality", CFG["d"])
    assert all(torch.equal(agg[k], sd0[k]) for k in sd0 if key_modality(k) in ("A", "V"))
    print("2) aggregation checks passed")

    # 3) decoupling (linear model): fused logits == sum of g_m for every mask
    if CFG["model_type"] == "linear":
        mm = build_model(CFG).eval()
        xb = {m: DATA["train"][m][:16] for m in MODS}
        for pat in ["TAV", "TA", "T", "AV"]:
            mask = tuple(m in pat for m in MODS)
            lf, lm = mm({m: xb[m] for i, m in enumerate(MODS) if mask[i]}, mask)
            assert torch.allclose(lf, sum(lm.values()), atol=1e-6)
        print("3) fused == sum_m g_m for all masks passed")

    # 4) BIO loss equals a manual evaluation of Eq. (12)
    c_full = [c for c in cl_all if len(c["mods"]) == 3][0]
    b = c_full["idx"][:32]; xb = {m: DATA["train"][m][b] for m in MODS}; yb = DATA["train"]["y"][b]
    mm = build_model(CFG).eval()
    L, info = local_loss(mm, xb, yb, (True, True, True), {**CFG, "bio_mode": "paper"}, True)
    with torch.no_grad():
        lf, lm = mm(xb, (True, True, True))
        ar = torch.arange(len(yb), device=yb.device)
        ce = lambda l: -F.log_softmax(l, 1)[ar, yb].mean()
        rho = {m: F.softmax(lm[m], 1)[ar, yb].sum() for m in MODS}
        rbar = sum(rho.values()) / 3
        I = {m: 1.0 if rho[m] < rbar else 0.0 for m in MODS}
        man = max(1 - sum(I.values()), 0) * ce(lf) + sum(I[m] * (ce(lf) + ce(lm[m])) for m in MODS)
    assert torch.allclose(L.detach(), man, atol=1e-5), (float(L), float(man))
    print("4) BIO loss == Eq.(12) passed | I flags:", info["I"])

    # 5) BIO gradient flows to the weak modality's parameters
    mm = build_model(CFG).train(); L, info = local_loss(mm, xb, yb, (True, True, True), CFG, True); L.backward()
    weak = [m for m, v in info["I"].items() if v == 1.0]
    for m in weak:
        assert any(p.grad is not None and p.grad.abs().sum() > 0 for k, p in mm.named_parameters() if key_modality(k) == m)
    print("5) gradients reach weak modalities", weak, "passed")

    # 6) TCM: only the strong modality is perturbed, at roughly the probability p_m
    N = 4000
    xb = {m: torch.randn(N, 8, device=DEVICE) for m in MODS}
    rec = new_record(N); rec["valid"][:] = True; rec["ok"][:] = True
    rec["p"][:, 0] = 0.9; rec["p"][:, 1] = 0.2; rec["p"][:, 2] = 0.2
    out = apply_tcm(xb, torch.arange(N, device=DEVICE), MODS, rec, CFG)
    chg = {m: float((out[m] != xb[m]).any(1).float().mean()) for m in MODS}
    exp = min(3 * 0.9 / (0.9 + 0.2 + 0.2), CFG["tcm_lambda"]) / CFG["tcm_lambda"]
    assert chg["A"] == 0 and chg["V"] == 0 and abs(chg["T"] - exp) < 0.04, (chg, exp)
    rec["ok"][:, 0] = False                                         # incorrectly-classified samples are never perturbed
    out = apply_tcm(xb, torch.arange(N, device=DEVICE), MODS, rec, CFG)
    assert float((out["T"] != xb["T"]).any(1).float().mean()) == 0
    print(f"6) TCM check passed | perturbed fraction of T = {chg['T']:.3f} (expected {exp:.3f})")
    # 7) FedProx is really active: a large mu must keep local parameters closer to the global model
    def _dist(mu):
        set_seed(123)
        m_, _ = local_train(sd0, cl, DATA, CFG, False, False, mu, CFG["lr"])
        sdm = m_.state_dict()
        return float(sum(((sdm[k] - sd0[k]) ** 2).sum() for k in sd0 if sd0[k].is_floating_point()) ** 0.5)
    d0, d1 = _dist(0.0), _dist(10.0)
    assert d1 < d0, f"FedProx has no effect (distance mu=0: {d0:.4f}, mu=10: {d1:.4f})"
    assert CFG["prox_mu"] > 0, "CFG['prox_mu'] must be > 0 for A1b_fedprox to differ from A1_fedavg"
    print(f"7) FedProx active passed | distance to global: mu=0 -> {d0:.4f}, mu=10 -> {d1:.4f}")
    print("\nAll unit checks passed.")

## 7. Experiment suite (baselines + ablations)

| Name | Description |
|---|---|
| `Centralized` | One client, all train data, full TAV, CE only (reference) |
| `Centralized_IBML` | Same with BIO + TCM: tests the paper's method *without* FL / missing modalities |
| `A0_local` | Local-only, CE (evaluated under each client's own mask) |
| `A1_fedavg` | FedAvg over all parameters |
| `A1b_fedprox` | A1 + proximal term |
| `A2_modagg` | Modality-aware aggregation |
| `A3_modagg_bio` | A2 + BIO (paper loss `L_mib`) |
| `A3b_modagg_tcm` | A2 + TCM only |
| `A4_modagg_ibml` | A2 + BIO + TCM (full IBML in FL) |
| `A5_ibml_nowarmup` | A4 without warm-up (paper's ablation) |
| `A6_bio_plusCE` | A3 with `CE_f + λ·L_mib` (the earlier notebook's loss form) |
| `A1c_modagg_fedprox` | A2 + proximal term: FedProx with modality-aware aggregation |
| `A7_fedprox_ibml` | **FedProx + IBML**: A2 + proximal term + BIO + TCM |

In [ ]:
EXPS = {
    "A1_fedavg":        dict(agg="fedavg"),
    "A1b_fedprox":      dict(agg="fedavg", mu=None),
    "A2_modagg":        dict(agg="modality"),
    "A3_modagg_bio":    dict(agg="modality", use_bio=True),
    "A3b_modagg_tcm":   dict(agg="modality", use_tcm=True),
    "A4_modagg_ibml":   dict(agg="modality", use_bio=True, use_tcm=True),
    "A5_ibml_nowarmup": dict(agg="modality", use_bio=True, use_tcm=True, warmup=False),
    "A6_bio_plusCE":    dict(agg="modality", use_bio=True, cfg_over=dict(bio_mode="plus_ce")),
    "A1c_modagg_fedprox": dict(agg="modality", mu=None),                                   # FedProx + modality-aware aggregation
    "A7_fedprox_ibml":  dict(agg="modality", mu=None, use_bio=True, use_tcm=True),       # FedProx + IBML (BIO + TCM)
}

def run_suite(names, cfg, seeds, include_local=True, include_central=True, tag="main"):
    rows = []
    for seed in seeds:
        clients = build_clients(DATA, cfg, seed)
        if include_central:
            n = len(DATA["train"]["y"])
            all_c = [dict(id=0, pattern="TAV", mods=MODS, mask=(True, True, True),
                          idx=torch.arange(n, device=DEVICE), n=n, pos_frac=0)]
            rows.append(run_federated(DATA, all_c, cfg, agg="fedavg", seed=seed, name="Centralized"))
            rows.append(run_federated(DATA, all_c, cfg, agg="fedavg", use_bio=True, use_tcm=True,
                                      seed=seed, name="Centralized_IBML"))
        if include_local:
            rows.append(run_local_only(DATA, clients, cfg, seed=seed))
        for n in names:
            t0 = time.time()
            rows.append(run_federated(DATA, clients, cfg, seed=seed, name=n, **EXPS[n]))
            print(f"[{tag}] seed={seed} {n:18s} best_val={rows[-1]['best_val']:.3f} round={rows[-1]['best_round']:2d} ({time.time()-t0:.0f}s)")
    return rows

def summarise(rows):
    recs = []
    for r in rows:
        t = r["test"]
        full = t.get("TAV", t.get("own_mask"))
        miss = [v["macro_f1"] for c, v in t.items() if c not in ("TAV", "own_mask")]
        rec = dict(method=r["name"], seed=r["seed"], acc_TAV=full["acc"], f1_TAV=full["macro_f1"],
                   bal_acc_TAV=full["bal_acc"], wf1_TAV=full["weighted_f1"],
                   rec_pos_TAV=full["rec_pos"], rec_neg_TAV=full["rec_neg"],
                   f1_missing_mean=float(np.mean(miss)) if miss else np.nan,
                   delta_miss=(full["macro_f1"] - float(np.mean(miss))) if miss else np.nan,
                   disc_ratio=r.get("bal", {}).get("ratio", np.nan),
                   rounds_to_thr=np.nan if r["rounds_to_thr"] is None else r["rounds_to_thr"],
                   comm_MB=r["comm_MB"])
        for c in ALL_CONDS[1:]:
            rec[f"f1_{c}"] = t[c]["macro_f1"] if c in t else np.nan
        recs.append(rec)
    df = pd.DataFrame(recs)
    g = df.drop(columns="seed").groupby("method", sort=False)
    mean, std = g.mean(), g.std()
    return df, mean.round(4).astype(str) + " ± " + std.fillna(0).round(4).astype(str)

### Plot helpers

In [ ]:
def plot_convergence(rows):
    plt.figure(figsize=(7, 4))
    for n in dict.fromkeys(r["name"] for r in rows if r["history"]):
        H = np.array([[h["val_score"] for h in r["history"]] for r in rows if r["name"] == n])
        plt.plot(range(1, H.shape[1] + 1), H.mean(0), label=n)
    plt.xlabel("communication round"); plt.ylabel("val macro-F1 (mean over 7 conditions)")
    plt.legend(fontsize=7); plt.grid(alpha=.3); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/convergence.png", dpi=150); plt.show()

def plot_robustness(rows):
    names = [n for n in dict.fromkeys(r["name"] for r in rows) if n != "A0_local"]
    M = np.array([[np.mean([r["test"][c]["macro_f1"] for r in rows if r["name"] == n]) for c in ALL_CONDS] for n in names])
    w = 0.8 / len(names); x = np.arange(len(ALL_CONDS))
    plt.figure(figsize=(10, 4))
    for i, n in enumerate(names): plt.bar(x + i * w, M[i], w, label=n)
    plt.xticks(x + 0.4 - w / 2, ALL_CONDS); plt.ylabel("test macro-F1"); plt.legend(fontsize=6, ncol=2); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/robustness.png", dpi=150); plt.show()

def plot_balance(rows):
    names = [n for n in dict.fromkeys(r["name"] for r in rows if "bal" in r)]
    vals = [np.mean([r["bal"]["ratio"] for r in rows if r["name"] == n]) for n in names]
    plt.figure(figsize=(8, 3.5)); plt.bar(names, vals); plt.axhline(1.0, color="k", lw=.8, ls="--")
    plt.xticks(rotation=45, ha="right", fontsize=7); plt.ylabel("discrepancy ratio max(rho)/min(rho)")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/balance.png", dpi=150); plt.show()

## 3. Per-dataset pipeline

For each dataset: load → unit checks → **LR selection on validation** (plain CE inside the federated setup, then fixed for every method) → baselines and ablations over seeds → plots → optional sensitivity sweep → save. Test data is never used for any selection; checkpoints inside a run are chosen on validation.

In [ ]:
def tune_lr():
    # Choose the learning rate on VALIDATION with plain-CE training (no IBML, no FedProx); then keep it fixed for ALL methods.
    # "federated": modality-aware FL on the real client split (seed 0): matches how the methods are trained.
    # "central"  : one centralized client (can pick an LR that is too small for FL on tiny clients).
    n = len(DATA["train"]["y"])
    if CFG["tune_regime"] == "federated":
        clients, agg = build_clients(DATA, CFG, CFG["seeds"][0]), "modality"
    else:
        clients = [dict(id=0, pattern="TAV", mods=MODS, mask=(True, True, True),
                        idx=torch.arange(n, device=DEVICE), n=n, pos_frac=0)]
        agg = "fedavg"
    scores = {}
    for lr in CFG["lr_grid"]:
        r = run_federated(DATA, clients, {**CFG, "lr": lr}, agg=agg, seed=0, name=f"lr={lr}")
        scores[lr] = r["best_val"]
        print(f"  lr={lr:<6} best val score = {scores[lr]:.4f}  (best round {r['best_round']})")
    CFG["lr"] = max(scores, key=scores.get)
    print("  -> selected lr =", CFG["lr"], f"({CFG['tune_regime']} regime)")
    if CFG["lr"] in (min(CFG["lr_grid"]), max(CFG["lr_grid"])):
        print("  NOTE: the selected LR is at the edge of lr_grid; extend the grid.")
    return scores

SENS_METHODS = ["A2_modagg", "A4_modagg_ibml", "A7_fedprox_ibml"]

def tune_mu():
    # Choose the FedProx coefficient on VALIDATION (seed 0) with FedProx + modality-aware aggregation and plain CE
    # (no IBML). The chosen value is used by every FedProx variant (A1b, A1c, A7). mu=0 is evaluated only for comparison.
    clients = build_clients(DATA, CFG, CFG["seeds"][0]); scores = {}
    for mu in CFG["mu_grid"]:
        r = run_federated(DATA, clients, CFG, agg="modality", mu=mu, seed=0, name=f"mu={mu}")
        scores[mu] = r["best_val"]
        print(f"  mu={mu:<6} best val score = {scores[mu]:.4f}  (best round {r['best_round']})")
    pos = {m: v for m, v in scores.items() if m > 0}
    CFG["prox_mu"] = max(pos, key=pos.get)
    print("  -> selected mu =", CFG["prox_mu"])
    if 0.0 in scores and scores[0.0] >= pos[CFG["prox_mu"]]:
        print("  NOTE: mu=0 scored at least as well as the best positive mu on validation: "
              "FedProx gives no validation gain on this dataset/partition.")
    if CFG["prox_mu"] in (min(pos), max(pos)):
        print("  NOTE: the selected mu is at the edge of the grid; extend mu_grid.")
    return scores

def _run_extra_checks():
    # 8) with a single available modality BIO must reduce to plain CE (no invented cross-modal term)
    c1 = [c for c in build_clients(DATA, CFG, 0) if len(c["mods"]) == 1][0]
    b = c1["idx"][:16]; xb = {m: DATA["train"][m][b] for m in c1["mods"]}; yb = DATA["train"]["y"][b]
    mm = build_model(CFG).eval()
    L_bio, _ = local_loss(mm, xb, yb, c1["mask"], CFG, True)
    L_ce, _ = local_loss(mm, xb, yb, c1["mask"], CFG, False)
    assert torch.allclose(L_bio, L_ce), "BIO must equal CE for single-modality clients"
    # 9) both partitions cover the training set exactly once; balanced sizes differ by at most 1
    y = DATA["train"]["y"].cpu().numpy()
    for kind in ("dirichlet", "balanced"):
        cl = build_clients(DATA, {**CFG, "partition": kind}, 0)
        ids = torch.cat([c["idx"] for c in cl]).cpu().numpy()
        assert len(ids) == len(np.unique(ids)) == len(y), f"{kind}: partition overlaps or misses samples"
    sizes = [c["n"] for c in build_clients(DATA, {**CFG, "partition": "balanced"}, 0)]
    assert max(sizes) - min(sizes) <= 1, f"balanced partition sizes differ: {sizes}"
    # 10) label handling: -1/+1 files map to 0/1; neutral sentiment scores are dropped
    keep, yy = _to_labels(np.array([-1, 1, 1, -1, 1]), "binary")
    assert keep.all() and yy.tolist() == [0, 1, 1, 0, 1]
    keep, yy = _to_labels(np.array([-2.0, 0.0, 1.5]), "sentiment")
    assert keep.tolist() == [True, False, True] and yy.tolist() == [0, 1]
    print("8-10) single-modality BIO == CE, partitions cover the training set, label mapping: passed")

def run_sensitivity():
    presets = {"none": ["TAV"], "mild": ["TAV", "TAV", "TAV", "TA", "TV"],
               "moderate": list(CFG["client_patterns"]),
               "severe": ["T", "A", "V", "TA", "T", "V", "AV", "A", "T", "TAV"]}
    rows = []
    for tag, pats in presets.items():
        rows += [dict(r, name=f"{r['name']}|missing={tag}") for r in
                 run_suite(SENS_METHODS, {**CFG, "client_patterns": pats}, CFG["seeds"],
                           include_local=False, include_central=False, tag=f"sens-{tag}")]
    for a in [100.0, 0.5, 0.1]:
        rows += [dict(r, name=f"{r['name']}|alpha={a}") for r in
                 run_suite(SENS_METHODS, {**CFG, "alpha": a}, CFG["seeds"],
                           include_local=False, include_central=False, tag=f"sens-a{a}")]
    rows += [dict(r, name=f"{r['name']}|partition=balanced") for r in
             run_suite(SENS_METHODS, {**CFG, "partition": "balanced"}, CFG["seeds"],
                       include_local=False, include_central=False, tag="sens-balanced")]
    _, sens_sum = summarise(rows)
    sens_sum.to_csv(f"{OUT_DIR}/sensitivity_summary.csv")
    display(sens_sum[["acc_TAV", "f1_TAV", "f1_missing_mean", "delta_miss", "disc_ratio"]])

def save_artifacts(rows):
    with open(f"{OUT_DIR}/config.json", "w") as f: json.dump(CFG, f, indent=2, default=str)
    for s in CFG["seeds"]:
        cl = build_clients(DATA, CFG, s)
        client_balance_report(cl).to_csv(f"{OUT_DIR}/client_balance_seed{s}.csv", index=False)
        pd.DataFrame([dict(client=c["id"], pattern=c["pattern"], n=c["n"], pos_frac=c["pos_frac"],
                           train_indices=c["idx"].cpu().tolist()) for c in cl]).to_json(
            f"{OUT_DIR}/clients_seed{s}.json", orient="records")
    with open(f"{OUT_DIR}/main_results_full.json", "w") as f:
        json.dump([dict(name=r["name"], seed=r["seed"], best_round=r["best_round"], best_val=r["best_val"],
                        test=r["test"], bal=r.get("bal"), comm_MB=r["comm_MB"], history=r["history"])
                   for r in rows], f, default=float, indent=1)

def paired_report(raw_df, cols=("f1_TAV", "f1_missing_mean"),
                  pairs=(("A3_modagg_bio", "A2_modagg"), ("A4_modagg_ibml", "A2_modagg"),
                         ("A4_modagg_ibml", "A3_modagg_bio"), ("A1b_fedprox", "A1_fedavg"), ("A1c_modagg_fedprox", "A2_modagg"),
                         ("A7_fedprox_ibml", "A4_modagg_ibml"), ("A7_fedprox_ibml", "A1c_modagg_fedprox"),
                         ("Centralized_IBML", "Centralized"))):
    # Per-seed paired differences (same seed = same client split). Far more informative than comparing mean +- std.
    out = []
    for a, b in pairs:
        A = raw_df[raw_df.method == a].set_index("seed"); B = raw_df[raw_df.method == b].set_index("seed")
        if A.empty or B.empty: continue
        for c in cols:
            d = (A[c] - B[c]).dropna()
            if len(d): out.append(dict(comparison=f"{a} - {b}", metric=c, mean_diff=round(float(d.mean()), 4),
                                       per_seed=[round(float(x), 4) for x in d], seeds_better=f"{int((d > 0).sum())}/{len(d)}"))
    return pd.DataFrame(out)

FEDPROX_FAMILY = ("A1b_fedprox", "A1c_modagg_fedprox", "A7_fedprox_ibml")

def rank_by_validation(rows):
    # The "best method" is chosen on VALIDATION score only. Test numbers are shown for reporting, never for selection.
    df = pd.DataFrame([dict(method=r["name"], seed=r["seed"], val_score=r["best_val"],
                            test_f1_TAV=r["test"]["TAV"]["macro_f1"])
                       for r in rows if "TAV" in r["test"] and r["name"].startswith("A")])      # federated methods only
    g = df.groupby("method", sort=False).agg(val_mean=("val_score", "mean"), val_std=("val_score", "std"),
                                              test_f1_TAV_mean=("test_f1_TAV", "mean"))
    g = g.sort_values("val_mean", ascending=False).round(4)
    g.insert(0, "rank_by_val", range(1, len(g) + 1))
    best = g.index[0]
    print(f"Best federated method by validation: {best}")
    fp = [m for m in FEDPROX_FAMILY if m in g.index]
    print("FedProx-family ranks:", {m: int(g.loc[m, 'rank_by_val']) for m in fp},
          "| FedProx+IBML (A7) is best:" , best == "A7_fedprox_ibml",
          "| best FedProx-family member:", min(fp, key=lambda m: g.loc[m, 'rank_by_val']) if fp else None)
    return g

def run_dataset(name):
    done = f"{ROOT_OUT}/{name}/DONE_v4"
    if os.path.exists(done) and not FORCE_RERUN:
        print(f"[{name}] already finished -> skipping (set FORCE_RERUN=True to redo)")
        return pd.read_csv(f"{ROOT_OUT}/{name}/main_summary.csv", index_col=0)
    print("\n" + "=" * 70 + f"\n  {name}\n" + "=" * 70)
    use_dataset(name)
    print(client_balance_report(build_clients(DATA, CFG, seed=CFG["seeds"][0])).to_string(index=False))
    _run_unit_checks()
    _run_extra_checks()
    if CFG["tune_lr"]:
        print("Selecting learning rate on validation ...")
        tune_lr()
    if CFG["tune_mu"]:
        print("Selecting FedProx mu on validation ...")
        tune_mu()
    rows = run_suite(list(EXPS), CFG, CFG["seeds"], tag=name)
    raw_df, summary_df = summarise(rows)
    raw_df.to_csv(f"{OUT_DIR}/main_raw.csv", index=False)
    summary_df.to_csv(f"{OUT_DIR}/main_summary.csv")
    display(summary_df)
    ranking = rank_by_validation(rows); ranking.to_csv(f"{OUT_DIR}/ranking_by_validation.csv"); display(ranking)
    paired = paired_report(raw_df); paired.to_csv(f"{OUT_DIR}/paired_differences.csv", index=False)
    print("Paired per-seed differences (positive = first method better):"); display(paired)
    plot_convergence(rows); plot_robustness(rows); plot_balance(rows)
    if CFG["run_sensitivity"]:
        run_sensitivity()
    save_artifacts(rows)
    open(f"{OUT_DIR}/DONE_v4", "w").write("ok")
    return summary_df

## 4. Run the datasets

In [ ]:
RESULTS, FAILED = {}, {}
for _name in DATASETS_TO_RUN:
    if _name in RESULTS: continue
    try:
        RESULTS[_name] = run_dataset(_name)
    except Exception as e:                       # one failing dataset (e.g. a download quota) must not stop the others
        FAILED[_name] = str(e)
        print(f"\n!!! {_name} failed: {str(e)[:600]}\n")
print("finished:", list(RESULTS), "| failed:", list(FAILED))

## 5. Cross-dataset summary (mean ± std over seeds, test set)

In [ ]:
def cross_dataset_table(metrics=("f1_TAV", "f1_missing_mean", "acc_TAV")):
    frames = {}
    for n in DATASETS_TO_RUN:
        p = f"{ROOT_OUT}/{n}/main_summary.csv"
        if os.path.exists(p):
            frames[n] = pd.read_csv(p, index_col=0)[list(metrics)]
    if not frames:
        print("no finished datasets yet"); return None
    big = pd.concat(frames, axis=1)
    big.to_csv(f"{ROOT_OUT}/all_datasets_summary.csv")
    return big

cross_dataset_table()

## 6. Reporting guidance

* Present A4 as **"IBML applied in a federated, missing-modality setting"**; the paper never studies FL, and computing `ρ_m` on small, label-skewed client batches is your extension.
* The FedProx μ is tuned on validation with plain CE and then shared by `A1b`, `A1c` and `A7`; report the chosen μ and the validation scores for each μ (they are printed per dataset). Check `client_balance_seed*.csv` and mention tiny or single-class clients when you describe the partition.
* **FedProx + IBML (`A7`)** is reported alongside its parts: compare it with `A4` (IBML without the proximal term) and `A1c` (proximal term without IBML) in `paired_differences.csv`. Only call the combination useful if it beats both on most seeds and the validation ranking agrees.
* `Centralized_IBML` vs `Centralized` shows whether the IBML loss and TCM help *at all* on each dataset before FL enters. If they do not, FL results will not isolate their effect either.
* Compare methods by **mean ± std over seeds on the test set**. Model selection (checkpoint and learning rate) used validation only.
* **MUStARD** (138 test samples) and **MOSI** (686) give noisy macro-F1; differences of a point or two are not meaningful. Say so.
* The learning rate is tuned with plain CE and then reused by every method, which can slightly favour the CE baselines; state this.
* Encoders are MLPs on pre-extracted, time-pooled features, so absolute numbers are **not comparable** to published ResNet/BiGRU results.
* `disc_ratio` (max ρ / min ρ on test under full modalities) is the paper's balance indicator; closer to 1 means more balanced.